In [1]:
import os, csv, sqlite3, prettytable, warnings, pandas as pd

warnings.filterwarnings("ignore", category=SyntaxWarning)

%load_ext sql
%sql sqlite:///../data/databases/my_data1.db

con = sqlite3.connect("../data/databases/my_data1.db")
cur = con.cursor()

The 'toml' package isn't installed. To load settings from pyproject.toml or ~/.jupysql/config, install with: pip install toml

Connecting to 'sqlite:///../data/databases/my_data1.db'

In [2]:
df = pd.read_csv("../data/raw/Spacex.csv")
df.to_sql("SPACEXTBL", con, if_exists='replace', index=False,method="multi")

101

In [3]:
%sql DROP TABLE IF EXISTS SPACEXTABLE;

Running query in 'sqlite:///../data/databases/my_data1.db'

++
||
++
++

In [4]:
%sql create table SPACEXTABLE as select * from SPACEXTBL where Date is not null

Running query in 'sqlite:///../data/databases/my_data1.db'

++
||
++
++

## Task 1 - Displaying the names of the unique launch sites in the missions

In [5]:
%sql SELECT DISTINCT Launch_Site FROM SPACEXTABLE

Running query in 'sqlite:///../data/databases/my_data1.db'

Launch_Site
CCAFS LC-40
VAFB SLC-4E
KSC LC-39A
CCAFS SLC-40


## Task 2 - Displaying 5 records where launch sites begin with the string 'CCA'

In [6]:
%%sql
SELECT * 
FROM SPACEXTABLE 
WHERE Launch_Site LIKE 'CCA%' LIMIT 5;

Running query in 'sqlite:///../data/databases/my_data1.db'

Date,Time (UTC),Booster_Version,Launch_Site,Payload,PAYLOAD_MASS__KG_,Orbit,Customer,Mission_Outcome,Landing_Outcome
2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure (parachute)
2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel of Brouere cheese",0,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute)
2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2,525,LEO (ISS),NASA (COTS),Success,No attempt
2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500,LEO (ISS),NASA (CRS),Success,No attempt
2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677,LEO (ISS),NASA (CRS),Success,No attempt


## Task 3 - Displaying the total payload mass carried by boosters launched by NASA (CRS)

In [7]:
%sql SELECT SUM(PAYLOAD_MASS__KG_) AS "Total_Payload_NASA(CRS)" FROM SPACEXTABLE WHERE Customer = "NASA (CRS)"

Running query in 'sqlite:///../data/databases/my_data1.db'

Total_Payload_NASA(CRS)
45596


## Task 4 - Displaying average payload mass carried by booster version F9 v 1.1

In [8]:
%sql SELECT ROUND(AVG(PAYLOAD_MASS__KG_),2) AS "Average Payload by F9 v1.1" FROM SPACEXTABLE WHERE Booster_Version LIKE "F9 V1.1%"

Running query in 'sqlite:///../data/databases/my_data1.db'

Average Payload by F9 v1.1
2534.67


## Task 5 - Listing the date when the first successful landing outcome in ground pad was acheived

In [9]:
%sql SELECT MIN(Date) as "First Success(ground pad)" FROM SPACEXTABLE WHERE Landing_Outcome = 'Success (ground pad)' 

Running query in 'sqlite:///../data/databases/my_data1.db'

First Success(ground pad)
2015-12-22


## Task 6 - Listing the names of the boosters which have success in drone ship and have payload mass greater than 4000 but less than 6000

In [10]:
%%sql
SELECT Booster_Version AS 'Boosters w/ drone ship success, & 4000<Payload<6000 '
    FROM SPACEXTABLE
    WHERE Landing_Outcome = "Success (drone ship)" 
    AND PAYLOAD_MASS__KG_ BETWEEN 4000 AND 6000

Running query in 'sqlite:///../data/databases/my_data1.db'

"Boosters w/ drone ship success, & 4000<Payload<6000"
F9 FT B1022
F9 FT B1026
F9 FT B1021.2
F9 FT B1031.2


## Task 7 - Listing the total number of successful and failure mission outcomes

In [11]:
failure = %sql SELECT COUNT(*) FROM SPACEXTABLE WHERE Mission_Outcome NOT LIKE "%Success%"
success = %sql SELECT COUNT(*) FROM SPACEXTABLE WHERE Mission_Outcome LIKE "%Success%"

print(f'No. of Failures:{failure[0][0]}, \nNo. of Successes:{success[0][0]}')

Running query in 'sqlite:///../data/databases/my_data1.db'

Running query in 'sqlite:///../data/databases/my_data1.db'

No. of Failures:1, 
No. of Successes:100


## Task 8 - Listing all the booster versions that have carried the maximum payload mass, using a subquery with a suitable aggregate function

In [12]:
%%sql 
SELECT Booster_Version, PAYLOAD_MASS__KG_
FROM SPACEXTABLE 
WHERE PAYLOAD_MASS__KG_ = (SELECT MAX(PAYLOAD_MASS__KG_) FROM SPACEXTABLE);


Running query in 'sqlite:///../data/databases/my_data1.db'

Booster_Version,PAYLOAD_MASS__KG_
F9 B5 B1048.4,15600
F9 B5 B1049.4,15600
F9 B5 B1051.3,15600
F9 B5 B1056.4,15600
F9 B5 B1048.5,15600
F9 B5 B1051.4,15600
F9 B5 B1049.5,15600
F9 B5 B1060.2,15600
F9 B5 B1058.3,15600
F9 B5 B1051.6,15600


## Task 9 - Listing the records which will display the month names, failure landing_outcomes in drone ship ,booster versions, launch_site for the months in year 2015.

SQLLite does not support monthnames. So we need to use  substr(Date, 6,2) as month to get the months and substr(Date,0,5)='2015' for year.

In [13]:
%%sql
SELECT 
    substr(Date, 6, 2) as Month,
    Landing_Outcome,
    Booster_Version,
    Launch_Site
FROM SPACEXTABLE 
WHERE substr(Date, 1, 4) = '2015'
    AND Landing_Outcome = "Failure (drone ship)";

Running query in 'sqlite:///../data/databases/my_data1.db'

Month,Landing_Outcome,Booster_Version,Launch_Site
01,Failure (drone ship),F9 v1.1 B1012,CCAFS LC-40
04,Failure (drone ship),F9 v1.1 B1015,CCAFS LC-40


## Task 10 - Ranking the count of landing outcomes (such as Failure (drone ship) or Success (ground pad)) between the date 2010-06-04 and 2017-03-20, in descending order.

In [14]:
%%sql
SELECT 
    Landing_Outcome, 
    COUNT(Landing_Outcome) AS Outcome_Count 
FROM SPACEXTABLE 
WHERE 
    Date BETWEEN "2010-06-04" AND "2017-03-20"
GROUP BY Landing_Outcome
ORDER BY Outcome_Count DESC;

Running query in 'sqlite:///../data/databases/my_data1.db'

Landing_Outcome,Outcome_Count
No attempt,10
Success (drone ship),5
Failure (drone ship),5
Success (ground pad),3
Controlled (ocean),3
Uncontrolled (ocean),2
Failure (parachute),2
Precluded (drone ship),1


## Task 11 - Finding the total payload mass carried

In [15]:
%sql SELECT SUM(PAYLOAD_MASS__KG_) as Total_Payload_Mass from SPACEXTABLE 

Running query in 'sqlite:///../data/databases/my_data1.db'

Total_Payload_Mass
619967
